# Apagar bloques posteriores y conectar directo al classifier (2026-08-23)

Esto es lo que describiste con tu dibujo: si el parámetro es "bloque 2", se apaga todo lo que viene después del bloque 2 (bloques 3 y 4 ni siquiera corren) y el bloque 2 se conecta directo al `classifier`. Barrer el parámetro (bloque 1, bloque 2, bloque 3, bloque 4 = todos) es ver qué tan bien clasificaría el modelo si terminara ahí.

**Aviso honesto antes de programar nada:** esto es matemáticamente *lo mismo* que ya calculó `notebooks/17082026_logit_lens/logit_lens.ipynb`. Ahí, en vez de apagar bloques, "leíamos" el CLS después de cada bloque y lo proyectábamos por el clasificador — pero como el forward de un transformer es estrictamente secuencial (un bloque nunca modifica hacia atrás lo que ya calculó un bloque anterior), el CLS después del bloque 2 es exactamente el mismo tensor exista o no un bloque 3 y 4 corriendo después. Es la misma cantidad, calculada de dos formas distintas:

- **logit lens** (`logit_lens.ipynb`): corre el modelo completo, va "leyendo" el CLS en cada parada.
- **este notebook**: trunca el forward de verdad — los bloques posteriores al corte ni se ejecutan — y conecta ese punto directo al clasificador.

Vale la pena tenerlo como notebook aparte de todas formas: el código deja explícito "apagar bloques y conectar al classifier", que es como lo estás pensando, y sirve de base directa para la comparación baseline vs. penalizado. Abajo hay una celda que verifica numéricamente que ambas formas dan exactamente el mismo resultado — no es solo una afirmación, se comprueba.

In [ ]:
import json
import sys
from pathlib import Path

# Misma convención que el resto de notebooks/: vive 2 niveles bajo la raíz del proyecto.
project_root = Path.cwd().resolve().parents[1]
print(f"Project root: {project_root}")
sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision

from modules.model import ViTForClassfication
from modules.datasets import CIFAR10Dataset

torch.manual_seed(0)

## 1. Cargar el modelo baseline

Mismo checkpoint que en los otros tres notebooks (`logit_lens.ipynb`, `ablation.ipynb`, `head_contribution.ipynb`): `vit-with-15-epochs-CIFAR10`, para que los cuatro sean comparables entre sí. `map_location="cpu"` porque el checkpoint se guardó desde MPS.

In [ ]:
EXPERIMENT_NAME = "vit-with-15-epochs-CIFAR10"
exp_dir = project_root / "experimentation" / "experiments" / EXPERIMENT_NAME

with open(exp_dir / "config.json") as f:
    config = json.load(f)
with open(exp_dir / "metrics.json") as f:
    metrics = json.load(f)

model = ViTForClassfication(config)
state_dict = torch.load(exp_dir / "model_final.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

n_blocks = len(model.encoder.blocks)
print(f"Checkpoint: {EXPERIMENT_NAME}  (accuracy de entrenamiento: {metrics['accuracy']:.4f})")
print(f"Bloques: {n_blocks}")

## 2. Datos de test (CIFAR-10)

Mismo patrón que en los otros tres notebooks: apuntamos a `experimentation/data`, donde ya está descargado CIFAR-10.

In [ ]:
dataset = CIFAR10Dataset()
dataset.testset = torchvision.datasets.CIFAR10(
    root=str(project_root / "experimentation" / "data"),
    train=False,
    download=True,
    transform=dataset.test_transform,
)
classes = dataset.classes

N_EVAL = 500  # tamano del lote de evaluacion (CPU-friendly)
eval_images, eval_labels = dataset.get_samples_from_indices(range(N_EVAL), device="cpu", set="test")
print(f"Lote de evaluacion: {N_EVAL} imagenes de test")

## 3. La función: truncar el forward y conectar al classifier

`forward_truncated_at(model, x, cutoff_block_idx)`: corre los bloques `0..cutoff_block_idx` (inclusive) y **ni siquiera ejecuta** los que vienen después — se detiene ahí y pasa ese CLS directo por `self.classifier`. `cutoff_block_idx = -1` es el caso extremo: ningún bloque corre, solo los embeddings (referencia de qué tan bien clasifica el modelo sin haber pasado por ningún bloque — debería estar cerca del azar). `cutoff_block_idx = n_blocks - 1` es el modelo completo, el baseline normal.

In [ ]:
@torch.no_grad()
def forward_truncated_at(model, x, cutoff_block_idx):
    h = model.embedding(x)
    for i, block in enumerate(model.encoder.blocks):
        if i > cutoff_block_idx:
            break  # bloques posteriores al corte: apagados, ni se ejecutan
        h, _ = block(h, output_attentions=False)
    return model.classifier(h[:, 0, :])


@torch.no_grad()
def accuracy_of(logits, labels):
    return (logits.argmax(dim=-1) == labels).float().mean().item()

In [ ]:
# Sanity check 1: cutoff = ultimo bloque debe coincidir EXACTAMENTE con el forward normal.
with torch.no_grad():
    direct_logits, _ = model(eval_images, output_attentions=False)

full_logits = forward_truncated_at(model, eval_images, cutoff_block_idx=n_blocks - 1)
diff = (direct_logits - full_logits).abs().max().item()
print(f"Diferencia maxima (cutoff = ultimo bloque) vs forward normal: {diff:.2e}")
assert diff < 1e-5

baseline_accuracy = accuracy_of(direct_logits, eval_labels)
print(f"Accuracy baseline (n={N_EVAL}): {baseline_accuracy:.3f}")
print("OK - con el corte en el ultimo bloque, la funcion reproduce el forward normal.")

In [ ]:
# Sanity check 2: esto es lo importante para el punto de arriba -- comprobar que
# "truncar el forward de verdad" da EXACTAMENTE lo mismo que "leer el CLS
# intermedio" (el enfoque de logit_lens.ipynb), para cada bloque.
@torch.no_grad()
def logit_lens_style(model, x):
    """Igual que en logit_lens.ipynb: corre el modelo completo, va guardando
    el CLS despues de cada bloque, y proyecta cada uno con el clasificador."""
    h = model.embedding(x)
    cls_states = [h[:, 0, :]]  # capa 0: solo embeddings
    for block in model.encoder.blocks:
        h, _ = block(h, output_attentions=False)
        cls_states.append(h[:, 0, :])
    return [model.classifier(cls) for cls in cls_states]

logit_lens_logits = logit_lens_style(model, eval_images)  # indice 0 = embeddings, indice k = bloque k

max_diff_overall = 0.0
for cutoff in range(-1, n_blocks):
    truncated_logits = forward_truncated_at(model, eval_images, cutoff_block_idx=cutoff)
    d = (truncated_logits - logit_lens_logits[cutoff + 1]).abs().max().item()
    max_diff_overall = max(max_diff_overall, d)
    print(f"cutoff={cutoff:>2}: diff vs logit_lens_style = {d:.2e}")

assert max_diff_overall < 1e-5, "Truncar el forward y leer el CLS intermedio deberian dar exactamente lo mismo"
print("\nOK - confirmado: apagar los bloques posteriores da EXACTAMENTE lo mismo que leer el CLS intermedio.")
print("Es la misma cantidad que ya viste en logit_lens.ipynb, calculada de otra forma.")

## 4. Accuracy al cortar en cada bloque

Ahora sí, el barrido que pediste: `cutoff = 0` = solo bloque 1 corre (se apagan 2, 3 y 4); `cutoff = 1` = corren bloques 1 y 2 (se apagan 3 y 4); ... `cutoff = 3` = los 4 bloques (baseline). Incluimos también `cutoff = -1` (ni un bloque, solo embeddings) como referencia de piso.

In [ ]:
cutoffs = list(range(-1, n_blocks))
cutoff_accuracies = []
for cutoff in cutoffs:
    logits = forward_truncated_at(model, eval_images, cutoff_block_idx=cutoff)
    acc = accuracy_of(logits, eval_labels)
    cutoff_accuracies.append(acc)
    if cutoff == -1:
        desc = "solo embeddings (se apagan los 4 bloques)"
    else:
        off = list(range(cutoff + 2, n_blocks + 1))
        desc = f"corren bloques 1..{cutoff + 1}" + (f" (se apagan {off})" if off else " (= baseline, ningun bloque apagado)")
    print(f"cutoff={cutoff:>2}: accuracy = {acc:.3f}  -- {desc}")

fig, ax = plt.subplots(figsize=(7, 4))
xlabels = ["solo\nembeddings"] + [f"bloque {i+1}" for i in range(n_blocks)]
ax.plot(range(len(cutoffs)), cutoff_accuracies, marker="o", linewidth=2, color="#b65a22")
ax.axhline(baseline_accuracy, color="green", linestyle="--", linewidth=1, label=f"baseline (los 4 bloques) = {baseline_accuracy:.3f}")
ax.axhline(1 / len(classes), color="gray", linestyle=":", linewidth=1, label="azar (1/10)")
ax.set_xticks(range(len(cutoffs)))
ax.set_xticklabels(xlabels)
ax.set_xlabel("bloque conectado directo al classifier (bloques posteriores apagados)")
ax.set_ylabel("accuracy")
ax.set_title(f"Apagar bloques posteriores ({EXPERIMENT_NAME}, n={N_EVAL})")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

## Para reflexionar / siguientes pasos

- Esta curva debería verse idéntica a la de accuracy-por-capa de `logit_lens.ipynb` (ya lo comprobamos numéricamente arriba) — dos formas de llegar al mismo lugar. Si en algún momento las dos notebooks te dan números distintos, algo se rompió en una de las dos implementaciones.
- La pregunta que sí es nueva acá, comparado con `logit_lens.ipynb`: ¿tiene sentido para la tesis mostrar esto como "apagar bloques" (lenguaje de ablación/intervención) en vez de "leer capas intermedias" (lenguaje de logit lens)? Son el mismo número, pero el marco importa para cómo lo explicas en el jurado — "apagar" sugiere causalidad/intervención, "leer" sugiere solo observación pasiva.

**Siguiente paso real:** repetir esto con `vit-with-10-epochs-interpretable-CIFAR10` y comparar las dos curvas (baseline vs. penalizado) en el mismo gráfico — igual que se propuso para `ablation.ipynb` y `head_contribution.ipynb`. Con las cuatro notebooks corriendo sobre los dos checkpoints ya está armado el bloque A (feature study) completo de `CLAUDE.md`.